# Flower Distributed FL — MNIST Orchestrator

This notebook runs the same distributed federation as `Orchestrator.ipynb`,
but trains a **CNN on MNIST** instead of logistic regression on synthetic data.

## Model architecture

```
Input (1×28×28)
  Conv2d(1→32, k=3) → ReLU
  Conv2d(32→64, k=3) → ReLU → MaxPool2d(2) → Dropout(0.5)
  Flatten → Linear(9216→128) → ReLU → Dropout(0.5)
  Linear(128→10)   ← 10 digit classes
```

Each SuperNode trains on a **non-overlapping slice of the MNIST training set**
(30 000 samples per node with 2 nodes). MNIST is downloaded automatically on
first run to `~/.flwr_demo/mnist/`.

## Before running

1. Install PyTorch in your environment if not already present:
   ```bash
   pip install torch torchvision
   ```
2. On the **SuperLink machine**: `bash start_superlink.sh`
3. On each **SuperNode machine**: `bash start_supernode.sh <SUPERLINK_IP>`
4. Set `SUPERLINK_IP` below and run all cells.

---
## 0 - Imports & helpers

In [1]:
import os
import re
import socket
import subprocess
import sys
import time
from pathlib import Path

# tomli (read) and tomli_w (write) are installed as Flower dependencies.
# On Python 3.11+ tomllib is built-in; tomli_w is always needed for writing.
try:
    import tomllib
except ImportError:
    import tomli as tomllib  # type: ignore[no-redef]  # Python 3.10
import tomli_w

# Root of the fl_app package (relative to this notebook)
FL_APP_DIR = Path("fl_app_mnist").resolve()
PYPROJECT = FL_APP_DIR / "pyproject.toml"

# Locate the flwr CLI binary next to the active Python interpreter.
# This works correctly inside any activated virtual environment.
FLWR_BIN = str(Path(sys.executable).parent / "flwr")

print(f"Python      : {sys.executable}")
print(f"flwr binary : {FLWR_BIN}")
print(f"App dir     : {FL_APP_DIR}")
assert Path(FLWR_BIN).exists(), f"flwr not found at {FLWR_BIN} — wrong kernel?"
assert PYPROJECT.exists(), "pyproject.toml not found — is FL_APP_DIR correct?"

Python      : /home/leno/miniconda3/envs/flower_ai/bin/python
flwr binary : /home/leno/miniconda3/envs/flower_ai/bin/flwr
App dir     : /home/leno/FL/flower/FL_dist_demo/fl_app_mnist


In [2]:
def run_command(cmd: list[str], cwd: Path | None = None) -> int:
    """Run a command and stream its stdout/stderr to the notebook output."""
    print("$ " + " ".join(cmd))
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(cwd) if cwd else None,
        env=env,
    ) as proc:
        for line in proc.stdout:  # type: ignore[union-attr]
            print(line, end="", flush=True)
    return proc.returncode


def check_port_open(host: str, port: int, timeout: float = 2.0) -> bool:
    """Return True if a TCP connection to host:port succeeds."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def update_federation_address(address: str) -> None:
    """Update the distributed SuperLink address in ~/.flwr/config.toml.

    Flower >= 1.15 stores federation settings in ~/.flwr/config.toml
    under [superlink.distributed], not in pyproject.toml.
    """
    flwr_config = Path.home() / ".flwr" / "config.toml"
    if not flwr_config.exists():
        raise FileNotFoundError(
            f"{flwr_config} not found. Run a local-sim first so Flower creates it."
        )
    with open(flwr_config, "rb") as f:
        config = tomllib.load(f)
    config.setdefault("superlink", {}).setdefault("distributed", {})["address"] = address
    config["superlink"]["distributed"]["insecure"] = True
    with open(flwr_config, "wb") as f:
        tomli_w.dump(config, f)
    print(f"Updated ~/.flwr/config.toml: [superlink.distributed] address = {address}")


---
## 1 — Configuration

Set `SUPERLINK_IP` to the IP of the machine running `start_superlink.sh`.  
Leave it as `127.0.0.1` if everything runs on the same machine.

In [3]:
# ── Edit these values ──────────────────────────────────────────────────────
SUPERLINK_IP    = "127.0.0.1"  # IP of the SuperLink machine
NUM_ROUNDS      = 5            # federated rounds
MIN_CLIENTS     = 2            # minimum connected SuperNodes
NUM_PARTITIONS  = 2            # must match number of SuperNodes
MAX_SAMPLES     = 500          # training samples per node (500 ≈ 8 batches, ~10s/round on CPU)
# ───────────────────────────────────────────────────────────────────────────

SUPERLINK_FLEET_PORT     = 9092
SUPERLINK_SERVERAPP_PORT = 9093

print(f"SuperLink Fleet API  : {SUPERLINK_IP}:{SUPERLINK_FLEET_PORT}")
print(f"SuperLink ServerApp  : {SUPERLINK_IP}:{SUPERLINK_SERVERAPP_PORT}")
print(f"Rounds               : {NUM_ROUNDS}")
print(f"Min clients          : {MIN_CLIENTS}")
print(f"Partitions           : {NUM_PARTITIONS}")
print(f"Samples/node         : {MAX_SAMPLES}  (~{MAX_SAMPLES//64} batches/round)")

SuperLink Fleet API  : 127.0.0.1:9092
SuperLink ServerApp  : 127.0.0.1:9093
Rounds               : 5
Min clients          : 2
Partitions           : 2
Samples/node         : 500  (~7 batches/round)


---
## 2 — Quick local simulation (no external machines needed)

Runs 2 virtual SuperNodes in-process. MNIST will be downloaded on first run (~11 MB).
Each epoch takes longer than the synthetic demo — expect ~1-2 min per round on CPU.

In [4]:
print("Running local MNIST simulation (federation: local-sim) ...")
print("Note: MNIST download on first run, ~1-2 min per round on CPU.\n")
rc = run_command(
    [
        FLWR_BIN, "run", str(FL_APP_DIR), "local-sim",
        "--run-config", f"num-server-rounds={NUM_ROUNDS}",
        "--run-config", f"min-clients={MIN_CLIENTS}",
        "--run-config", f"num-partitions={NUM_PARTITIONS}",
        "--run-config", f"max-samples={MAX_SAMPLES}",
        "--federation-config", f"num-supernodes={MIN_CLIENTS}",
    ]
)
if rc == 0:
    print("\n\u2713 Local simulation completed successfully.")
else:
    print(f"\n\u2717 Simulation failed (exit code {rc}).")

Running local MNIST simulation (federation: local-sim) ...
Note: MNIST download on first run, ~1-2 min per round on CPU.

$ /home/leno/miniconda3/envs/flower_ai/bin/flwr run /home/leno/FL/flower/FL_dist_demo/fl_app_mnist local-sim --run-config num-server-rounds=5 --run-config min-clients=2 --run-config num-partitions=2 --run-config max-samples=500 --federation-config num-supernodes=2
⚠️ Warning: `options.` fields in the SuperLink connection configuration are deprecated. Use `--federation-config` with `flwr run` instead. Alternatively, permanently set your simulation configuration via `flwr federation simulation-config`.
⚠️ Warning: `--federation-config` was provided, so deprecated `options.` entries from the SuperLink connection in your Flower configuration will be ignored.
🎊 Successfully started run 13078801618339066351

✓ Local simulation completed successfully.


---
## 3 — Distributed federation

### 3a — Verify connectivity

Make sure the SuperLink is reachable before launching the run.

In [5]:
# Check SuperLink reachability
fleet_ok    = check_port_open(SUPERLINK_IP, SUPERLINK_FLEET_PORT)
serverapp_ok = check_port_open(SUPERLINK_IP, SUPERLINK_SERVERAPP_PORT)

print(f"Fleet API    {SUPERLINK_IP}:{SUPERLINK_FLEET_PORT}  → {'✓ reachable' if fleet_ok else '✗ NOT reachable'}")
print(f"ServerApp    {SUPERLINK_IP}:{SUPERLINK_SERVERAPP_PORT}  → {'✓ reachable' if serverapp_ok else '✗ NOT reachable'}")

if not (fleet_ok and serverapp_ok):
    print("\n⚠  SuperLink is not reachable. Did you run `bash start_superlink.sh` on the server machine?")
else:
    print("\nSuperLink is up. Make sure the SuperNodes are also running before the next cell.")

Fleet API    127.0.0.1:9092  → ✓ reachable
ServerApp    127.0.0.1:9093  → ✓ reachable

SuperLink is up. Make sure the SuperNodes are also running before the next cell.


### 3b — Update federation address and run

The cell below patches `pyproject.toml` with the correct `SUPERLINK_IP`,  
then calls `flwr run fl_app/ distributed`.  
The SuperLink will distribute the app bundle to the connected SuperNodes automatically.

In [6]:
import re

address = f"{SUPERLINK_IP}:{SUPERLINK_SERVERAPP_PORT}"
update_federation_address(address)

print("\nSubmitting MNIST run to SuperLink...\n")
result = subprocess.run(
    [
        FLWR_BIN, "run", str(FL_APP_DIR), "distributed",
        "--run-config", f"num-server-rounds={NUM_ROUNDS}",
        "--run-config", f"min-clients={MIN_CLIENTS}",
        "--run-config", f"num-partitions={NUM_PARTITIONS}",
        "--run-config", f"max-samples={MAX_SAMPLES}",
    ],
    capture_output=True, text=True,
)
print(result.stdout)

try:
    match = re.search(r"run (\d+)", result.stdout)
    if not match:
        print("\u26a0 Could not extract run ID — check output above.")
    else:
        run_id = match.group(1)
        print(f"Run ID: {run_id}\n")
        print("Streaming logs until run completes...\n")
        rc = run_command([FLWR_BIN, "log", run_id, "distributed", "--stream"])
        if rc == 0:
            print("\n\u2713 Distributed MNIST run completed.")
        else:
            print(f"\n\u2717 Log stream ended with exit code {rc}.")
except Exception as e:
    print(f"Error: {e}")


Updated ~/.flwr/config.toml: [superlink.distributed] address = 127.0.0.1:9093

Submitting MNIST run to SuperLink...

🎊 Successfully started run 10563607877341497328

Run ID: 10563607877341497328

Streaming logs until run completes...

$ /home/leno/miniconda3/envs/flower_ai/bin/flwr log 10563607877341497328 distributed --stream
INFO :      Starting logstream for run_id `10563607877341497328`
INFO :      Start `flwr-serverapp` process
INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 2 clients (out of 2)
INFO :      aggregate_fit: received 2 results and 0 failures
INFO :      configure_evaluate: strategy sampled 2 clients (out of 2)
INFO :      aggregate_evaluate: received 1 r

---
## 4 — Optional: run everything locally for testing

If you want to test the distributed path **without multiple machines**,  
this cell starts a SuperLink and two SuperNodes as background subprocesses  
on this machine, waits for them to register, then runs the federation.

In [ ]:
import os

procs: list[subprocess.Popen] = []

def _start(cmd: list[str], label: str) -> subprocess.Popen:
    p = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"Started {label} (pid {p.pid})")
    procs.append(p)
    return p

def stop_all():
    for p in procs:
        try:
            p.terminate()
        except Exception:
            pass
    procs.clear()
    print("All background processes stopped.")

BIN_DIR = Path(sys.executable).parent
SUPERLINK_BIN = str(BIN_DIR / "flower-superlink")
SUPERNODE_BIN = str(BIN_DIR / "flower-supernode")

# Start infrastructure
_start([SUPERLINK_BIN, "--insecure", "--database", ":flwr-in-memory:"], "SuperLink")
time.sleep(2)
_start([SUPERNODE_BIN, "--insecure",
        "--superlink", "127.0.0.1:9092",
        "--clientappio-api-address", "0.0.0.0:9094",
        "--max-retries", "0"], "SuperNode-1")
time.sleep(2)
_start([SUPERNODE_BIN, "--insecure",
        "--superlink", "127.0.0.1:9092",
        "--clientappio-api-address", "0.0.0.0:9096",
        "--max-retries", "0"], "SuperNode-2")
time.sleep(3)

# Submit and stream
update_federation_address("127.0.0.1:9093")
print("\nSubmitting MNIST run...\n")
result = subprocess.run(
    [
        FLWR_BIN, "run", str(FL_APP_DIR), "distributed",
        "--run-config", f"num-server-rounds={NUM_ROUNDS}",
        "--run-config", "min-clients=2",
        "--run-config", f"num-partitions={NUM_PARTITIONS}",
    ],
    capture_output=True, text=True,
)
print(result.stdout)

try:
    match = re.search(r"run (\d+)", result.stdout)
    if not match:
        print("\u26a0 Could not extract run ID.")
    else:
        run_id = match.group(1)
        print(f"Run ID: {run_id}\n")
        print("Streaming logs...\n")
        rc = run_command([FLWR_BIN, "log", run_id, "distributed", "--stream"])
        status = "\u2713 completed" if rc == 0 else f"\u2717 failed (exit {rc})"
        print(f"\nRun {status}.")
finally:
    stop_all()
